# Limits of Diversification

Diversification benefits are conditional on cross-asset dependence. This notebook studies industry return correlations through time and examines how average correlation behaves around market stress.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import risk_kit as rk

returns = rk.get_ind_returns().dropna(how='all')
WINDOW = 36
print(f'{returns.shape[1]} industries | {len(returns)} monthly observations')

## 1. Industry breadth and return history

The dataset contains 30 value-weighted US industry portfolios. Each industry is a distinct return series, but industries are exposed to common macroeconomic and financial shocks.

In [ ]:
returns.iloc[-60:].plot(figsize=(12, 6), legend=False, alpha=0.55)
plt.title('Monthly industry returns: final 60 observations')
plt.ylabel('Monthly return')
plt.tight_layout()

## 2. Rolling pairwise correlations

For each month, calculate the correlation matrix over the trailing 36-month window. Average only the unique off-diagonal correlations; including the diagonal would mechanically pull the mean upward.

In [ ]:
rolling_corr = returns.rolling(WINDOW).corr()
industry_count = returns.shape[1]
upper_i, upper_j = np.triu_indices(industry_count, k=1)
mean_corr = rolling_corr.groupby(level=0).apply(
    lambda matrix: matrix.to_numpy()[upper_i, upper_j].mean()
).dropna()
ax = mean_corr.plot(figsize=(11, 5), color='navy')
ax.set(title=f'Mean pairwise industry correlation ({WINDOW}-month window)', ylabel='Average correlation')
ax.grid(alpha=.25)
plt.tight_layout()

## 3. Trailing market return and correlation

The equal-weighted industry basket below is a transparent analytical proxy, not the official US market portfolio. It is shown alongside average pairwise correlation to illustrate how diversification can weaken when common shocks dominate.

In [ ]:
equal_weight_return = returns.mean(axis=1)
trailing_return = equal_weight_return.rolling(WINDOW).apply(
    lambda x: rk.annualize_rets(pd.Series(x), 12), raw=False
).dropna()
comparison = pd.concat([trailing_return.rename('Trailing annualised return'),
                        mean_corr.rename('Mean pairwise correlation')], axis=1).dropna()
ax = comparison.plot(secondary_y='Mean pairwise correlation', figsize=(11, 5))
ax.set_title('Trailing return and average industry correlation')
ax.grid(alpha=.25)
plt.tight_layout()
print('Correlation between the two plotted series:', comparison.corr().iloc[0, 1])

## 4. Optional market-cap weighting extension

The original class exercise uses industry firm counts and average firm size to construct industry market capitalisations. The two companion files are not part of the committed dataset snapshot. Add the monthly files named ind30_m_nfirms.csv and ind30_m_size.csv to data/ to run this section; no values are fabricated when they are absent.

In [ ]:
try:
    firms = rk.get_ind_nfirms()
    average_size = rk.get_ind_size()
except FileNotFoundError as exc:
    print(f'Optional market-cap analysis skipped: {exc}')
else:
    cap = firms * average_size
    total_cap = cap.sum(axis=1).replace(0, np.nan)
    cap_weights = cap.div(total_cap, axis=0)
    aligned_returns = returns.reindex(index=cap.index, columns=cap.columns)
    cap_weighted_returns = (cap_weights * aligned_returns).sum(axis=1).dropna()
    cap_weights[['Steel', 'Fin']].plot(figsize=(10, 4), title='Steel and Financials share of industry market capitalisation')
    plt.tight_layout()
    cap_weighted_returns.plot(figsize=(10, 4), title='Market-cap-weighted 30-industry proxy returns')
    plt.tight_layout()

## Interpretation

- Correlation is time-varying; diversification measured in calm periods may overstate protection in crises.
- The average correlation is a summary and hides dispersion across individual industry pairs.
- An equal-weighted industry basket is not the broad market index.
- The optional market-cap reconstruction depends on companion datasets and their definitions.